In [4]:
import numpy as np
import pandas as pd
import seaborn as sns


from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_absolute_error
from nltk.cluster.kmeans import KMeansClusterer
from nltk.cluster.util import euclidean_distance
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from ucimlrepo import fetch_ucirepo
# fetch dataset
support2 = fetch_ucirepo(id=880)

import umap.umap_ as umap
import matplotlib.pyplot as plt

from models import *
from my_encodings import *

In [38]:
# --------------------------- Initialize dataset ------------------------------
# data (as pandas dataframes)
features = support2.data.features
targets = support2.data.targets

# join together as 1 df for EDA 
df = features.join(targets)
df = df.drop(columns=['death', 'hospdead', 'adlp', 'adls', 'scoma', 'totmcst', 'totcst', 'sps', 'aps', 'hday', 'adlsc', 'prg2m', 'prg6m', 'charges', 'dzgroup', 'dnr', 'dnrday', 'urine']) # drop other target features

# drop columns where target = NaN
print(f'Before dropping NaNs in target: {df.shape[0]}')
df = df = df.dropna(subset=['sfdm2'])
print(f'After dropping NaNs in target: {df.shape[0]}')

# check remaining columns
print(df.columns)
print(len(df.columns))

Before dropping NaNs in target: 9105
After dropping NaNs in target: 7705
Index(['age', 'sex', 'dzclass', 'num.co', 'edu', 'income', 'avtisst', 'race',
       'surv2m', 'surv6m', 'diabetes', 'dementia', 'ca', 'meanbp', 'wblc',
       'hrt', 'resp', 'temp', 'pafi', 'alb', 'bili', 'crea', 'sod', 'ph',
       'glucose', 'bun', 'sfdm2'],
      dtype='object')
27


Pipeline for encoding, scaling, and imputing NaNs

https://www.geeksforgeeks.org/machine-learning/handling-missing-data-with-knn-imputer/

for numerical features
   1) impute
   2) scale


for categorical features
   1) encode to numerical
   2) impute

In [39]:
X = df.drop(columns=['sfdm2'])
y = df['sfdm2']

# split into training vs test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

In [43]:
# ----------------------------- process numerical features -----------------------------
def num_feature_pre_processing(training_set: pd.DataFrame, test_set: pd.DataFrame):
    numerical_feats = training_set.select_dtypes(include=['number']).columns.tolist()
    numerical_df_train = training_set[numerical_feats]
    numerical_df_test = test_set[numerical_feats]

    # impute
    imputer = KNNImputer(n_neighbors=2) # initialize imputer 

    # training
    x_train = imputer.fit_transform(numerical_df_train) # fit and transform training set 
    x_train = pd.DataFrame(x_train, columns=numerical_df_train.columns, index=numerical_df_train.index)

    # testing
    x_test = imputer.transform(numerical_df_test)
    x_test = pd.DataFrame(x_test, columns=numerical_df_test.columns, index=numerical_df_test.index)

    # scale
    scaler = StandardScaler()

    # training
    x_train_scaled = scaler.fit_transform(x_train)

    # testing
    x_test_scaled = scaler.transform(x_test)

    return x_train, x_test


In [44]:
XS_train, XS_test = num_feature_pre_processing(X_train, X_test)

In [ ]:
# ------------------------------ process categorical features ---------------------------------


categorical_df = df.drop(columns=numerical_feats)